Python notebook to convert gliner saved results to Label studio annotations.

## Data and Libraries

In [1]:
import json
import uuid
import os
import re

import json
from datetime import datetime, timezone

from collections import defaultdict, Counter

import multiset_multicover as mm

### Helper Functions

In [3]:
def extract_paper_id(filename):
    match = re.match(r"\d+_(\d+)_", filename)
    return match.group(1) if match else "unknown"

def convert_to_labelstudio_format(data, paper_id, model_version="gliner-community/gliner_medium-v2.5"):
    if not isinstance(data, list):
        raise TypeError("Expected data to be a list of dictionaries, but got: {}".format(type(data)))
    
    labelstudio_data = []
    
    for item in data:
        if not isinstance(item, dict) or "entities" not in item:
            continue
        
        predictions = {
            "model_version": model_version,
            "score": 0.5,  # Default score for the overall prediction
            "result": []
        }
        
        for entity in item.get("entities", []):
            if not isinstance(entity, dict):
                continue
            
            predictions["result"].append({
                "id": str(uuid.uuid4())[:8],
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": entity.get("start", 0),
                    "end": entity.get("end", 0),
                    "score": entity.get("score", 0.0),
                    "text": entity.get("text", ""),
                    "labels": [entity.get("label", "Unknown")]
                }
            })
        
        item["paper_id"] = paper_id
        labelstudio_data.append({"data": item, "predictions": [predictions]})
    
    return labelstudio_data

def process_directory(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for filename in os.listdir(input_dir):
        if filename.endswith(".json"):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            paper_id = extract_paper_id(filename)
            
            with open(input_path, "r", encoding="utf-8") as infile:
                try:
                    data = json.load(infile)
                except json.JSONDecodeError:
                    print(f"Skipping {filename}: Invalid JSON format.")
                    continue
            
            try:
                transformed_data = convert_to_labelstudio_format(data, paper_id)
            except TypeError as e:
                print(f"Skipping {filename}: {e}")
                continue
            
            with open(output_path, "w", encoding="utf-8") as outfile:
                json.dump(transformed_data, outfile, indent=4)
                
def format_for_label_studio(alldata, project_id=20, start_task_id=0, start_prediction_id=0):
    """
    Wraps pre-formatted data and predictions into the full Label Studio task format.
    Handles a list of lists (papers) as input.

    Args:
        alldata (list of lists): A list of papers, where each paper is a list of sentence dicts.
        project_id (int): The ID of the Label Studio project these tasks belong to.
        start_task_id (int): The starting ID for the main tasks.
        start_prediction_id (int): The starting ID for the prediction objects.

    Returns:
        list: A list of tasks formatted for Label Studio import.
    """
    formatted_tasks = []
    
    # Initialize a global counter for unique IDs across all papers
    task_counter = 0

    for paper in alldata:
        for item in paper:
            # Use the global counter to ensure unique IDs
            task_id = start_task_id + task_counter
            prediction_id = start_prediction_id + task_counter

            # Get the current time in the required ISO format with 'Z'
            now_utc = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

            if not item.get('predictions'):
                continue
                
            source_prediction = item['predictions'][0]

            # Build the new, fully-structured task dictionary
            new_task = {
                "id": task_id,
                "data": item['data'],
                "annotations": [],
                "predictions": [
                    {
                        "id": prediction_id,
                        "result": source_prediction['result'],
                        "model_version": source_prediction.get('model_version'),
                        "created_ago": "0 minutes",
                        "score": source_prediction.get('score'),
                        "cluster": None,
                        "neighbors": None,
                        "mislabeling": 0,
                        "created_at": now_utc,
                        "updated_at": now_utc,
                        "model": None,
                        "model_run": None,
                        "task": task_id,
                        "project": project_id
                    }
                ]
            }
            formatted_tasks.append(new_task)
            
            # Increment the global counter for the next task
            task_counter += 1

    return formatted_tasks

def merge_annotations(formatted_tasks, annots_data):
    """
    Merges annotations from a Label Studio export into a list of formatted tasks.

    Args:
        formatted_tasks (list): The output from format_for_label_studio.
        annots_data (list): The original data from the Label Studio export.

    Returns:
        list: The formatted_tasks list with annotations added where they match.
    """
    # Create a fast lookup map from the annotations data
    annotations_map = {
        (task['data']['paper_id'], task['data']['sentence_id']): task.get('annotations', [])
        for task in annots_data
        if 'data' in task and 'paper_id' in task['data'] and 'sentence_id' in task['data']
    }

    # Iterate through the tasks and add the annotations
    for task in formatted_tasks:
        lookup_key = (task['data']['paper_id'], task['data']['sentence_id'])
        
        # If a matching annotation is found in the map, update the task
        if lookup_key in annotations_map:
            task['annotations'] = annotations_map[lookup_key]
            
    return formatted_tasks

def update_entity_labels(file_path):
    """
    Loads a Label Studio JSON file, remaps specific entity labels,
    and saves the file back.

    Args:
        file_path (str): The path to the JSON file to be updated.
    """
    # Define the remapping rules for old labels to new labels
    label_remap_rules = {
        "Artefact": "Intellectual Artefact",
        "Astronomical Object": "Location"
    }
    
    print(f"Loading data from '{file_path}'...")
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: The file '{file_path}' is not a valid JSON file.")
        return

    update_count = 0

    # This helper function will perform the update on a list of results
    def process_results(results):
        nonlocal update_count
        for result_item in results:
            if result_item.get('type') == 'labels' and 'value' in result_item and 'labels' in result_item['value']:
                # The 'labels' field is a list, e.g., ["Artefact"]
                # We update it in place
                original_labels = result_item['value']['labels']
                updated_labels = []
                for label in original_labels:
                    # If the label is in our rules, use the new value, otherwise keep the old one
                    if label in label_remap_rules:
                        updated_labels.append(label_remap_rules[label])
                        update_count += 1
                    else:
                        updated_labels.append(label)
                result_item['value']['labels'] = updated_labels

    # Iterate through every task in the dataset
    for task in data:
        # 1. Update labels within the 'annotations' section
        if 'annotations' in task:
            for annotation in task['annotations']:
                if 'result' in annotation:
                    process_results(annotation['result'])
        
        # 2. Update labels within the 'predictions' section
        if 'predictions' in task:
            for prediction in task['predictions']:
                if 'result' in prediction:
                    process_results(prediction['result'])

    if update_count > 0:
        print(f"Successfully updated {update_count} label occurrences.")
        print(f"Saving updated data back to '{file_path}'...")
        
        # Save the modified data back to the same file with pretty printing
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
            
        print("File saved successfully.")
    else:
        print("No labels matching the remapping rules were found. The file was not changed.")

def create_shorthand_string(entity_types):
    """Creates a single string of entity type shorthands from a list of full names."""
    shorthands = []
    for entity in entity_types:
        words = entity.split(' ')
        parts = [word[:3].capitalize() for word in words]
        shorthand = ''.join(parts)
        shorthands.append(shorthand)
    return '_'.join(shorthands)

def remove_annots(labelstudio_data):
    """
    Removes all annotations from a list of Label Studio tasks.

    This function iterates through each task and replaces the value of the
    'annotations' key with an empty list. It modifies the data in-place.

    Args:
        labelstudio_data (list): A list of task dictionaries from Label Studio.

    Returns:
        list: The same list of tasks with the annotations cleared.
    """
    for task in labelstudio_data:
        if 'annotations' in task:
            task['annotations'] = []
    return labelstudio_data

## GLiNER to Label Studio

In [6]:
# Example usage (Change accordingly)
input_directory = "RESULTS/GLINER_MEDIUM_V25_V1_PSS/"
output_directory = "RESULTS/GLINER_MEDIUM_V25_V1_PSS_LS/"
process_directory(input_directory, output_directory)


## Label Studio Merge

First, we connect all the GLiNER predictions into a single list of jsons.

In [13]:
input_directory = "RESULTS/GLINER_MEDIUM_V25_V1_PSS_LS/"
output_file_path = "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile.json"

output_file = []
for filename in os.listdir(input_directory):
    if filename.endswith(".json"):
        input_path = os.path.join(input_directory, filename)
             
        with open(input_path, "r", encoding="utf-8") as infile:
            try:
                data = json.load(infile)
            except json.JSONDecodeError:
                print(f"Skipping {filename}: Invalid JSON format.")
                continue
        
        output_file.append(data)
        
with open(output_file_path, "w", encoding="utf-8") as outfile:
    json.dump(output_file, outfile, indent=4)

Next, we want to update the data with arbitrary annotations:

In [20]:
annots_DIR = "DATA/project-6-at-2025-10-31-07-06-066d28f7.json"
alldata_DIR = "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile.json"

with open(annots_DIR, "r") as f:
    annots_data = json.load(f)

with open(alldata_DIR, "r") as f:
    alldata_data = json.load(f)

First, let's convert GLiNER annotated data to simulate Label Studio format:

In [23]:
# 2. Perform the transformation
final_data = format_for_label_studio(alldata_data)

# 3. Print the result in a readable JSON format
# print(json.dumps(final_data, indent=2))

# 4. (Optional) Save the result to a file for import
alldata_ls_DIR = alldata_DIR.split(".json")[0]+"_lsformat.json"
with open(alldata_ls_DIR, 'w') as f:
    json.dump(final_data, f, indent=2)

print(f"\nTransformation complete. See the output above or in '{alldata_ls_DIR}'.")


Transformation complete. See the output above or in 'RESULTS/gliner_medium_v25_v1_pss_ls_singlefile_lsformat.json'.


In [ ]:
alldata_ls_DIR = "RESULTS/gliner_medium_v25_v1_pss_ls_singlefile_lsformat.json"
with open(alldata_ls_DIR, "r") as f:
    alldata_ls = json.load(f)

# Step 2: Merge the existing annotations into the newly formatted tasks
final_merged_data = merge_annotations(alldata_ls, annots_data)

Save data ready for loading:

In [26]:
final_merged_data_DIR = "RESULTS/gliner_med_v25_v1_anotsready.json"
with open(final_merged_data_DIR, 'w') as f:
    json.dump(final_merged_data, f, indent=2)

Note that from v0 to v1 there have been few changes, adding few new entity types: "Asset, Physical Artefact and Policy" and two remapings: "Astronomical Object"->"Location" ("Astronomical Object" is deleted) and "Artefact"->"Intellectual Artefact". Thus, annotations should have these updated, especially "Artefact" and "Astronomical Object":

In [ ]:
final_merged_data_DIR = "RESULTS/gliner_med_v25_v1_anotsready.json"
# Run the function to update the labels
update_entity_labels(final_merged_data_DIR)

Loading data from 'RESULTS/gliner_med_v25_v1_anotsready.json'...
Successfully updated 403 label occurrences.
Saving updated data back to 'RESULTS/gliner_med_v25_v1_anotsready.json'...
File saved successfully.


Finally, this (gliner_med_v25_v1_anotsready) can be loaded into the Label Studio and used further for annotation.

## Annotator Data Creator

#### Selector based on greedy diversity preference

In [31]:
def select_diverse_annotated_samples(file_path, n_sample, entity_types_to_tag, diversity_bonus=1.0):
    """
    Selects a diverse set of annotated sentences using a greedy algorithm.

    The function prioritizes sentences that contribute examples of entity types
    that are furthest from their `n_sample` target. It also adds a bonus
    for sentences that introduce new entity text, promoting diversity.

    Args:
        file_path (str): Path to the input JSON file (e.g., 'gliner_med_v25_v1_anotsready.json').
        n_sample (int): The target number of examples for each entity type. This is a soft limit.
        entity_types_to_tag (list): A list of entity type strings to focus on.
        diversity_bonus (float): A bonus added to a sentence's score for each new, unseen
                                 entity text it provides.

    Returns:
        list: A list of the selected task dictionaries in their original format.
    """
    print("--- Starting Sample Selection ---")
    
    # 1. Load data and filter for already-annotated tasks
    try:
        with open(file_path, 'r') as f:
            all_tasks = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []

    candidate_tasks = [
        task for task in all_tasks
        if task.get('annotations') and task['annotations'][0].get('result')
    ]
    print(f"Found {len(candidate_tasks)} sentences with existing annotations to select from.")

    # 2. Initialize tracking structures
    entity_counts = {entity: 0 for entity in entity_types_to_tag}
    entity_diversity_sets = {entity: set() for entity in entity_types_to_tag}
    
    selected_tasks = []
    # Use a set of IDs for efficient lookup of already selected tasks
    selected_task_ids = set()

    # 3. Main iterative loop
    while True:
        # Check if all entity counts have reached the n_sample target
        is_complete = all(count >= n_sample for count in entity_counts.values())
        if is_complete:
            print("\nAll entity types have reached the target count.")
            break

        best_task = None
        max_score = -1

        # 4. Score every sentence not yet chosen
        for task in candidate_tasks:
            if task['id'] in selected_task_ids:
                continue

            current_score = 0
            # Use a temporary set to see what new entity texts this sentence would add
            temp_new_texts = defaultdict(set)

            annotations = task.get('annotations', [{}])[0].get('result', [])
            for annotation in annotations:
                # Safely access the label
                try:
                    label = annotation['value']['labels'][0]
                    text = annotation['value']['text'].lower()
                except (KeyError, IndexError):
                    continue # Skip malformed annotations

                if label in entity_types_to_tag:
                    # Score based on "need": how far is this entity from its goal?
                    need_score = max(0, n_sample - entity_counts[label])
                    current_score += need_score

                    # Add diversity bonus for unseen entity text
                    if text not in entity_diversity_sets[label]:
                        current_score += diversity_bonus
                        temp_new_texts[label].add(text)

            if current_score > max_score:
                max_score = current_score
                best_task = task
        
        # 5. If a best task is found, add it to the selection
        if best_task:
            selected_tasks.append(best_task)
            selected_task_ids.add(best_task['id'])

            # Update counts and diversity sets based on the chosen task's content
            annotations = best_task.get('annotations', [{}])[0].get('result', [])
            for annotation in annotations:
                try:
                    label = annotation['value']['labels'][0]
                    text = annotation['value']['text'].lower()
                except (KeyError, IndexError):
                    continue

                if label in entity_types_to_tag:
                    entity_counts[label] += 1
                    entity_diversity_sets[label].add(text)
        else:
            # No task can improve the score (e.g., all remaining sentences only contain entities
            # that have already met their quota).
            print("\nNo more useful sentences could be found. Stopping selection.")
            break
            
    # 6. Print a final summary of the selection
    print("\n--- Selection Summary ---")
    print(f"Selected a total of {len(selected_tasks)} sentences.")
    print("Final counts per entity type:")
    for entity, count in entity_counts.items():
        diversity = len(entity_diversity_sets[entity])
        print(f"  - {entity}: {count}/{n_sample} examples ({diversity} unique instances)")
    
    return selected_tasks

# --- EXAMPLE USAGE ---

# Define your parameters
FILE_PATH = "RESULTS/gliner_med_v25_v1_anotsready.json"
N_SAMPLES_PER_TYPE = 50
ENTITY_TYPES_TO_TAG = ["Mathematical Expression", "Measuring Device", "Physical Phenomenon", "Quantity", "Time Period", "Satellite"]

# Run the selection function
selected_data = select_diverse_annotated_samples(
    file_path=FILE_PATH,
    n_sample=N_SAMPLES_PER_TYPE,
    entity_types_to_tag=ENTITY_TYPES_TO_TAG,
    diversity_bonus=5.0 # Give a moderate bonus for new entity texts
)
filename = create_shorthand_string(ENTITY_TYPES_TO_TAG)

# You can now save the selected data to a new file if needed
if selected_data:
    output_filename = f"RESULTS/{N_SAMPLES_PER_TYPE}_{filename}.json"
    print(f"\nSaving {len(selected_data)} selected sentences to '{output_filename}'...")
    with open(output_filename, 'w') as f:
        json.dump(selected_data, f, indent=2)
    print("Done.")

--- Starting Sample Selection ---
Found 932 sentences with existing annotations to select from.

All entity types have reached the target count.

--- Selection Summary ---
Selected a total of 352 sentences.
Final counts per entity type:
  - Mathematical Expression: 66/50 examples (57 unique instances)
  - Measuring Device: 79/50 examples (62 unique instances)
  - Physical Phenomenon: 190/50 examples (166 unique instances)
  - Quantity: 966/50 examples (732 unique instances)
  - Time Period: 369/50 examples (281 unique instances)
  - Satellite: 50/50 examples (22 unique instances)

Saving 352 selected sentences to 'RESULTS/50_MatExp_MeaDev_PhyPhe_Qua_TimPer_Sat.json'...
Done.


#### Selector based on multiset multicover problem

In [5]:
def select_samples_with_msc_api(file_path, n_sample, entity_types_to_tag):
    """
    Selects annotated sentences using the correct multiset-multicover API.

    This function uses the GreedyCoverInstance object to find a near-optimal
    set of sentences meeting the coverage requirements for each entity type.

    Args:
        file_path (str): Path to the input JSON file.
        n_sample (int): The target number of examples for each entity type.
        entity_types_to_tag (list): A list of entity type strings to focus on.

    Returns:
        list: A list of the selected task dictionaries in their original format.
    """
    print("--- Starting Sample Selection using the multiset-multicover API ---")

    # 1. Load data and filter for already-annotated tasks
    try:
        with open(file_path, 'r') as f:
            all_tasks = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []

    candidate_tasks = [
        task for task in all_tasks
        if task.get('annotations') and task['annotations'][0].get('result')
    ]
    print(f"Found {len(candidate_tasks)} sentences with existing annotations to select from.")

    # 2. Create the necessary mappings and initialize the solver instance
    # The library requires integer indices for elements, not strings.
    entity_to_idx = {entity: i for i, entity in enumerate(entity_types_to_tag)}
    num_entity_types = len(entity_types_to_tag)

    # Initialize the GreedyCoverInstance with the size of the element universe
    gci = mm.GreedyCoverInstance(num_entity_types)
    
    # This list will map the index of a multiset added to the solver
    # back to the original task data.
    task_map = []

    print("Preprocessing data and adding multisets to the solver instance...")
    for task in candidate_tasks:
        # Count occurrences of each target entity in this task's annotations
        entity_counts_in_task = Counter(
            ann['value']['labels'][0]
            for ann in task['annotations'][0]['result']
            if 'value' in ann and 'labels' in ann['value'] and ann['value']['labels'][0] in entity_types_to_tag
        )
        
        if entity_counts_in_task:
            # Convert the counted entities (strings) into the required format:
            # - A list of integer indices for the elements.
            # - A corresponding list of their multiplicities (counts).
            elements_in_task = []
            multiplicities_in_task = []
            for entity, count in entity_counts_in_task.items():
                elements_in_task.append(entity_to_idx[entity])
                multiplicities_in_task.append(count)

            # Add this task as a multiset to the solver instance
            gci.add_multiset(elements_in_task, multiplicity=multiplicities_in_task)
            
            # Keep track of the original task object at this index
            task_map.append(task)

    if not task_map:
        print("No tasks found containing the specified entity types.")
        return []

    # 3. Define the coverage requirements
    # A list where each entry corresponds to the coverage needed for an entity index.
    requirements = [n_sample] * num_entity_types
    print(f"\nSeeking to cover {num_entity_types} entity types with {n_sample} examples each.")

    # 4. Run the solver by calling the .cover() method
    print("Running the multiset-multicover solver...")
    selected_indices = gci.cover(requirements)
    print(f"Solver finished. Selected {len(selected_indices)} sentences.")

    # 5. Retrieve the selected tasks using the returned indices and the map
    selected_tasks = [task_map[i] for i in selected_indices]
    
    # 6. Print a final summary of the selection
    print("\n--- Selection Summary ---")
    final_counts = Counter()
    for task in selected_tasks:
        for ann in task['annotations'][0]['result']:
            try:
                label = ann['value']['labels'][0]
                if label in entity_types_to_tag:
                    final_counts[label] += 1
            except (KeyError, IndexError):
                continue

    print("Final counts per entity type:")
    for entity in entity_types_to_tag:
        count = final_counts.get(entity, 0)
        print(f"  - {entity}: {count}/{n_sample} examples")
    
    return selected_tasks

# --- EXAMPLE USAGE ---

# Define your parameters
FILE_PATH = "DATA/LABEL_STUDIO/project-28-at-2025-11-05-09-02-8f888278.json" # "RESULTS/gliner_med_v25_v1_anotsready.json"
N_SAMPLES_PER_TYPE = 50
ENTITY_TYPES_TO_TAG = ["Method", "Field of Study", "Intellectual Artefact"] # SMI i KB
# ["Mathematical Expression", "Measuring Device", "Physical Phenomenon", "Quantity", "Time Period", "Satellite"] # MG

# Run the new, API-correct selection function
selected_data = select_samples_with_msc_api(
    file_path=FILE_PATH,
    n_sample=N_SAMPLES_PER_TYPE,
    entity_types_to_tag=ENTITY_TYPES_TO_TAG
)

# Generate the filename from the entity types
filename_shorthand = create_shorthand_string(ENTITY_TYPES_TO_TAG)

# Remove annotations
selected_data = remove_annots(selected_data)

# Save the selected data to a new file
if selected_data:
    output_filename = f"RESULTS/mmp_{N_SAMPLES_PER_TYPE}_{filename_shorthand}.json"
    print(f"\nSaving {len(selected_data)} selected sentences to '{output_filename}'...")
    with open(output_filename, 'w') as f:
        json.dump(selected_data, f, indent=2)
    print("Done.")

--- Starting Sample Selection using the multiset-multicover API ---
Found 972 sentences with existing annotations to select from.
Preprocessing data and adding multisets to the solver instance...

Seeking to cover 3 entity types with 50 examples each.
Running the multiset-multicover solver...
Solver finished. Selected 44 sentences.

--- Selection Summary ---
Final counts per entity type:
  - Method: 72/50 examples
  - Field of Study: 50/50 examples
  - Intellectual Artefact: 61/50 examples

Saving 44 selected sentences to 'RESULTS/mmp_50_Met_FieOfStu_IntArt.json'...
Done.
